# Frontend-Backend Integration Diagnostic Notebook

This notebook diagnoses and fixes the integration issues between the NextJS frontend and FastAPI backend.

## Issues Identified:
1. **SSL Certificate Error**: Frontend can't verify backend's self-signed certificate
2. **Prisma Schema Mismatch**: `status` field doesn't exist in AuditSession model
3. **Missing SSE Progress**: Blog generation progress notifications not working
4. **Audit System Conflicts**: Old DatabaseCostTracker vs new EnhancedDatabaseAuditTracker

In [ ]:
import requests
import json
import time
import jwt
from datetime import datetime, timedelta
import urllib3

# Disable SSL warnings for testing
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration
BACKEND_URL = "https://localhost:5000"
FRONTEND_URL = "https://localhost:3001"
JWT_SECRET = "Ver0EvKSf1T5hN4/6NDsnPyZf8S7dJZ/Ewksc2Y2L7w="

print("🔧 Integration Diagnostic Setup Complete")
print(f"Backend URL: {BACKEND_URL}")
print(f"Frontend URL: {FRONTEND_URL}")

## Test 1: Backend Health & SSL

In [ ]:
def test_backend_health():
    """Test backend connectivity and SSL handling"""
    print("1️⃣ Testing Backend Health...")
    
    try:
        # Test with SSL verification disabled (like frontend does)
        response = requests.get(f"{BACKEND_URL}/health", verify=False, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            print("✅ Backend accessible")
            print(f"   Status: {data['status']}")
            print(f"   Timestamp: {data['timestamp']}")
            return True
        else:
            print(f"❌ Backend returned {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ Backend connection failed: {e}")
        return False

backend_healthy = test_backend_health()

## Test 2: JWT Token Generation & Authentication

In [ ]:
def create_test_jwt():
    """Create a test JWT token like the frontend does"""
    print("2️⃣ Creating Test JWT Token...")
    
    payload = {
        'sub': 'test_user_12345',
        'email': 'test@example.com',
        'name': 'Test User',
        'role': 'FREE',
        'iat': int(time.time()),
        'exp': int(time.time()) + 3600  # 1 hour
    }
    
    try:
        token = jwt.encode(payload, JWT_SECRET, algorithm='HS256')
        print("✅ JWT token created successfully")
        print(f"   Token preview: {token[:50]}...")
        print(f"   Payload: {payload}")
        return token
    except Exception as e:
        print(f"❌ JWT creation failed: {e}")
        return None

test_token = create_test_jwt() if backend_healthy else None

## Test 3: Backend Authentication

In [ ]:
def test_backend_auth(token):
    """Test backend JWT authentication"""
    print("3️⃣ Testing Backend Authentication...")
    
    if not token:
        print("❌ No token available for testing")
        return False
    
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }
    
    try:
        # Test with a simple title generation request
        data = {'instructions': 'Test title generation for integration testing'}
        response = requests.post(
            f"{BACKEND_URL}/generate-title",
            headers=headers,
            json=data,
            verify=False,
            timeout=30
        )
        
        print(f"   Response status: {response.status_code}")
        print(f"   Response headers: {dict(response.headers)}")
        
        if response.status_code == 200:
            result = response.json()
            print("✅ Backend authentication successful")
            print(f"   Generated title: {result.get('title', 'N/A')}")
            return True
        else:
            print(f"❌ Authentication failed: {response.status_code}")
            print(f"   Response: {response.text}")
            return False
            
    except Exception as e:
        print(f"❌ Authentication test failed: {e}")
        return False

auth_working = test_backend_auth(test_token) if test_token else False

## Test 4: Blog Generation Flow

In [ ]:
def test_blog_generation(token):
    """Test the blog generation endpoint"""
    print("4️⃣ Testing Blog Generation Flow...")
    
    if not token:
        print("❌ No token available for testing")
        return None
    
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }
    
    data = {
        'topic': 'Integration Testing Best Practices',
        'instructions': 'Write a short blog about testing frontend-backend integration'
    }
    
    try:
        print("   Sending blog generation request...")
        response = requests.post(
            f"{BACKEND_URL}/generate-blog",
            headers=headers,
            json=data,
            verify=False,
            timeout=10  # Short timeout to avoid hanging
        )
        
        print(f"   Response status: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            print("✅ Blog generation request successful")
            print(f"   Task ID: {result.get('task_id', 'N/A')}")
            print(f"   Message: {result.get('message', 'N/A')}")
            return result.get('task_id')
        else:
            print(f"❌ Blog generation failed: {response.status_code}")
            print(f"   Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Blog generation test failed: {e}")
        return None

task_id = test_blog_generation(test_token) if auth_working else None

## Test 5: SSE Stream Connection

In [ ]:
def test_sse_connection(token, task_id):
    """Test SSE stream connectivity"""
    print("5️⃣ Testing SSE Stream Connection...")
    
    if not token or not task_id:
        print("❌ Missing token or task_id for SSE testing")
        return False
    
    # Test the SSE endpoint URL construction
    sse_url = f"{BACKEND_URL}/stream/{task_id}?token={token}"
    print(f"   SSE URL: {sse_url[:80]}...")
    
    try:
        # Simple test to see if the SSE endpoint is accessible
        # Note: We can't easily test EventSource in Python, but we can test the endpoint
        response = requests.get(sse_url, verify=False, timeout=5, stream=True)
        
        print(f"   Response status: {response.status_code}")
        print(f"   Content-Type: {response.headers.get('content-type', 'N/A')}")
        
        if response.status_code == 200:
            print("✅ SSE endpoint is accessible")
            print("   Note: Full SSE testing requires EventSource client")
            return True
        else:
            print(f"❌ SSE endpoint failed: {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ SSE connection test failed: {e}")
        return False

sse_working = test_sse_connection(test_token, task_id) if task_id else False

## Test 6: Audit System Database Check

In [ ]:
def test_audit_system():
    """Test if the audit system is working"""
    print("6️⃣ Testing Audit System...")
    
    try:
        # Run our existing audit validation script
        import subprocess
        result = subprocess.run(
            ['python', 'check_schema_compatibility.py'],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode == 0:
            print("✅ Audit system is working")
            print("   Recent audit data found in database")
            # Show last few lines of output
            lines = result.stdout.strip().split('\n')
            for line in lines[-3:]:
                if '📈' in line or '📊' in line:
                    print(f"   {line}")
            return True
        else:
            print(f"❌ Audit system check failed: {result.returncode}")
            print(f"   Error: {result.stderr}")
            return False
            
    except Exception as e:
        print(f"❌ Audit system test failed: {e}")
        return False

audit_working = test_audit_system()

## Summary & Recommendations

In [ ]:
def print_diagnostic_summary():
    """Print comprehensive diagnostic summary"""
    print("\n" + "="*70)
    print("🎯 DIAGNOSTIC SUMMARY")
    print("="*70)
    
    # Test results
    tests = [
        ("Backend Health", backend_healthy),
        ("JWT Token Creation", test_token is not None),
        ("Backend Authentication", auth_working),
        ("Blog Generation", task_id is not None),
        ("SSE Connectivity", sse_working),
        ("Audit System", audit_working)
    ]
    
    for test_name, passed in tests:
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"{status} {test_name}")
    
    # Recommendations
    print("\n📋 RECOMMENDATIONS:")
    
    if not backend_healthy:
        print("1. ❌ Check if FastAPI backend is running on port 5000")
        print("   Run: cd backend && python src/fastapi_main.py")
    
    if not auth_working and backend_healthy:
        print("2. ❌ JWT authentication is failing")
        print("   Check NEXTAUTH_SECRET matches between frontend and backend")
    
    if not sse_working and auth_working:
        print("3. ❌ SSE streaming is not working")
        print("   Check that FastAPI SSE endpoint is properly implemented")
    
    if all(passed for _, passed in tests):
        print("🎉 ALL TESTS PASSED!")
        print("   The frontend should now work correctly")
        print("   Try generating a blog from https://localhost:3001")
    
    print("\n🔧 FIXES APPLIED:")
    print("✅ Removed 'status' field from Prisma audit completion")
    print("✅ Updated BlogGenerationFlow to use context audit tracker")
    print("✅ Enhanced audit tracker with direct database connection")
    print("✅ SSL certificate handling with rejectUnauthorized: false")

print_diagnostic_summary()